#📚 文系大学生による『実践データ分析100本ノック＆LLM活用20本ノック』学習ログ
**著： 岡野 菜月（四国大学文学部 / 28卒）**

**GitHub： https://github.com/NatsukiOkano**

##📝 このノートブックの概要
＊ 書籍「Python 実践データ分析100本ノック 第3版」（秀和システム新社刊）の学習過程を記録したものです。

＊ 最大の特徴は、「文学部の視点でIT・データ分析を再解釈する」という試みにあります。

＊ 非情報系の学習者が直面する「専門用語の壁」を取り払うため、すべての処理に対し、論理的かつ直感的な「完全独自解説」を記述しています。

##⚠️ 注意事項
- **著作権保護の観点から、データセットおよび実行結果は同梱しておりません。**

### 教材
下山輝昌・松田雄馬・三木孝行 著「Python 実践データ分析100本ノック 第3版」（秀和システム新社、2025）

# ２章　小売店のデータでデータ加工を行う１０本ノック

本章では、ある小売店の売上履歴と顧客台帳データを用いて、データ分析の素地となる「データの加工」を習得することが目的です。
実際の現場データは手入力のExcel等、決して綺麗なデータではない事が多いため、
データの揺れや整合性の担保など、汚いデータを取り扱うデータ加工を主体に進めて行きます。

### ノック１１：データを読み込んでみよう

In [ ]:
import pandas as pd
uriage_data = pd.read_csv('uriage.csv')
uriage_data.head()

In [ ]:
kokyaku_data = pd.read_excel('kokyaku_daicho.xlsx')
kokyaku_data.head()

In [ ]:
#リレーショナルデータベース（RDB）：データを複数の表（テーブル）形式で管理し、それらの表を互いに関連付けたデータベース。

### ノック１２：データの揺れを見てみよう

In [ ]:
uriage_data['item_name'].head()

#データの揺れ：同じ意味のデータなのに、書き方がバラバラで「別のもの」として認識されてしまっている状態。

In [ ]:
uriage_data['item_price'].head()

### ノック１３：データに揺れがあるまま集計しよう

In [ ]:
uriage_data['purchase_date'] = pd.to_datetime(uriage_data['purchase_date'])
uriage_data['purchase_month'] = uriage_data['purchase_date'].dt.strftime('%Y%m')
res = uriage_data.pivot_table(index='purchase_month', columns='item_name', aggfunc='size', fill_value=0)
res

#pd.to_datetime：「文字（object）」として読み込まれた日付を、計算や加工ができる『日付専用データ（datetime）』に作り変える命令。
#to：～へ変換する。

#dt.strftime：「dt」datetimeの略。「strftime」時間を、形式を整えた文字にする。「string format time」の略。string（文字）format（形式を整える）time（時間）
#決まり文句：「%Y」西暦（4桁）,「%y」西暦（2桁）,「%m」月（2桁）,「%d」日（2桁）,「%H」時（2桁）,「%M」分（2桁）
#例：「%Y」2025,「%y」25,「%m」02,「%d」03,「%H」15,「%M」31

#pd.pivot_table：クロス集計表（Excelの表）を作る。※pd.pivot_table(表の名前,index=,columns=,values=,aggfunc=)
#「index=」行ラベル（縦軸）
#「columns=」列ラベル（横軸）
#「values=」値（計算対象）
#「aggfunc=」値の集計設定（基礎統計量）
#よく使う基礎統計量：sum（合計）,mean（平均値）,std（標準偏差）,median（中央値）,count（個数計算※欠損値(NaN)含まない）,size（行数計算※欠損値(NaN)含む）

#fill_value：空欄を埋めるための値の設定。※引数（設定項目）
#fill_value=0：空欄のセルは「NaN」ではなく「0」を入れる。

In [ ]:
res = uriage_data.pivot_table(index='purchase_month', columns='item_name', values='item_price', aggfunc='sum', fill_value=0)
res

### ノック１４：商品名の揺れを補正しよう

In [ ]:
print(len(pd.unique(uriage_data['item_name'])))

#print：目に見えるように、外に出して表示すること。
#len：データの件数を数える。Excelのcount関数と同じ役割。
#pd.unique：重複を削ぎ落した「純粋な種類の一覧」を1行で表示させる。

In [ ]:
print(pd.unique(uriage_data['item_name']))

In [ ]:
uriage_data['item_name'] = uriage_data['item_name'].str.upper()
uriage_data['item_name'] = uriage_data['item_name'].str.replace('  ','')
uriage_data['item_name'] = uriage_data['item_name'].str.replace(' ','')
uriage_data.sort_values(by=['item_name'], ascending=True)

#str.upper：アルファベットを全て大文字に変換する。
#str.replace('古い文字','新しい文字')：古い文字を探して、新しい文字に入れ替える。

#sort_values：表のデータを、特定の項目（列）を基準に並び替える。※.sort_values(by=[' '],ascending=)
#by：「どの列を基準にするか」を指定。※引数（設定項目）
#ascending：降順（False=0）か昇順（True=1）どっちで並べ替えるか設定する。※引数（設定項目）

In [ ]:
print(pd.unique(uriage_data['item_name']))
print(len(pd.unique(uriage_data['item_name'])))

#print：目に見えるように、外に出して表示すること。
#len：データの件数を数える。Excelのcount関数と同じ役割。
#pd.unique：重複を削ぎ落した「純粋な種類の一覧」を1行で表示させる。

### ノック１５：金額欠損値の補完をしよう

In [ ]:
uriage_data.isnull().any(axis=0)

#.isnull：「欠損値(NaN)セル」があるかどうかチェックする。→True：ある False：ない
#any：どれか、ひとつでも
#axis：処理する向き　axis=0：縦（列）　axis=1：横（行）

In [ ]:
flg_is_null = uriage_data['item_price'].isnull()
for trg in list(uriage_data.loc[flg_is_null, 'item_name'].unique()):
    price = uriage_data.loc[(~flg_is_null) & (uriage_data['item_name'] == trg), 'item_price'].max()
    uriage_data.loc[(flg_is_null) & (uriage_data['item_name']==trg),'item_price'] = price
uriage_data.head()

#for trg in list：リストの中身を1つずつ取り出して、順番に処理する。
#df.loc[行ラベル,列ラベル]：ラベルで指定。ラベルを使って条件に合うデータを探す。
#df.iloc[]：整数で指定。整数を使って条件に合うデータを探す。

#「~」：否定演算子。「NOT（～ではない）」を意味する記号。※flg_is_null：変数。~flg_is_null：flg_is_nullの反対。
#「=」：代入演算子。代入。例）x = 100：xに100を代入。→これ以降、xは100として扱われる。
#「==」：比較演算子。比較。(判定)　例）x == 100：xの中身は100と同じ？→もしxが100ならTrueになる。

In [ ]:
uriage_data.isnull().any(axis=0)

In [ ]:
for trg in list(uriage_data['item_name'].sort_values().unique()):
    print(trg + 'の最大額：' + str(uriage_data.loc[uriage_data['item_name']==trg]['item_price'].max()) + 'の最少額：' + str(uriage_data.loc[uriage_data['item_name']==trg]['item_price'].min(skipna=False)))

#for trg in list：リストの中身を1つずつ取り出して、順番に処理する。
#trg：変数。

#sort_values：表のデータを、特定の項目（列）を基準に並び替える。※.sort_values(by=[' '],ascending=)
#by：「どの列を基準にするか」を指定。※引数（設定項目）
#ascending：降順（False=0）か昇順（True=1）どっちで並べ替えるか設定する。※引数（設定項目）

#str：文字列。(テキストデータ)　※1行でも「文字列」と呼ぶ。文字行とは呼ばない。
#loc[行ラベル,列ラベル]：特定の条件に当てはまるデータだけを書き換える。

#pd.unique：重複を削ぎ落した「純粋な種類の一覧」を1行で表示させる。

### ノック１６：顧客名の揺れを補正しよう

In [ ]:
uriage_data['customer_name'].head()

In [ ]:
kokyaku_data['顧客名'].head()

In [ ]:
kokyaku_data['顧客名'] = kokyaku_data['顧客名'].str.replace('　','')
kokyaku_data['顧客名'] = kokyaku_data['顧客名'].str.replace(' ','')
kokyaku_data['顧客名'].head()

### ノック１７：日付の揺れを補正しよう

In [ ]:
flg_is_serial = kokyaku_data['登録日'].astype('str').str.isdigit()
print(flg_is_serial.sum())

#.astype：データの「クラス(データ型)」を強制的に変換する。
#.isdigit()：「文字列（str）の中身が、0〜9の数字だけで構成されているか」をチェックする。

In [ ]:
fromSerial = pd.to_timedelta(kokyaku_data.loc[flg_is_serial, '登録日'].astype('float') - 2, unit='D') + pd.to_datetime('1900/1/1')
fromSerial

#pd.to_timedelta('')：文字（object）」として読み込まれた期間を、計算・加工可能な『特定の期間（timedelta）』に作り変える命令。
#pd.to_datetime('')：「文字（object）」として読み込まれた日付を、計算・加工可能な『特定の日付（datetime）』に作り変える命令。
#to：～へ変換する。

#loc[行ラベル,列ラベル]：特定の条件に当てはまるデータだけを書き換える。

#astype('クラス')：クラス(データ型)を強制的に変更する。
#【astype()でよく使うクラス↓】
#  int：整数
#  float：少数
#  str：文字（Python(言語そのもの)が呼ぶ名前。）※ str = object
#  object：文字（pandasが、列の種類を表示するときに使う名前。）

#unit：「数字を日付に変換するとき、その数字が『1につき何時間（または何日）分』なのか」をコンピュータに教えるための単位指定のコード。
#unitでよく使われる単位：「日」('D'),「秒」('s'),「ミリ秒」('ms'),「マイクロ秒」('us'),「ナノ秒」('ns')

In [ ]:
fromString = pd.to_datetime(kokyaku_data.loc[~flg_is_serial, '登録日'])
fromString

In [ ]:
kokyaku_data['登録日'] = pd.concat([fromSerial, fromString])
kokyaku_data

In [ ]:
kokyaku_data['登録年月'] = kokyaku_data['登録日'].dt.strftime('%Y%m')
rslt = kokyaku_data.groupby('登録年月').count()['顧客名']
print(rslt)
print(len(kokyaku_data))

In [ ]:
flg_is_serial = kokyaku_data['登録日'].astype('str').str.isdigit()
print(flg_is_serial.sum())

#.astype：「クラス(データ型)」を強制的に変換する。
#.isdigit()：「文字列（str）の中身が、0〜9の数字だけで構成されているか」をチェックする。

### ノック１８：顧客名をキーに２つのデータを結合(ジョイン)しよう

In [ ]:
join_data = pd.merge(uriage_data, kokyaku_data, left_on='customer_name', right_on='顧客名', how='left')
join_data = join_data.drop('customer_name', axis=1)
join_data

#pd.merge()：結合。正規化されてバラバラになったデータを、分析しやすいように「逆正規化（合体）」させる。
#pd.merge(左の表,右の表, left_on=, right_on=, how=)

#left_on：左側の表(テーブル)の使用する列名を指定。
#right_on：右側の表(テーブル)の使用する列名を指定。
#※left_on, right_on はそれぞれの表(テーブル)で使う列名が異なるときに使う。

#how：表(テーブル)の結合方法を指定。（※よく使う結合方法↓）
#how='left'：左の表を主役にする。右に一致するデータがなくても、左のデータはすべて残す。
#how='right'：右の表を主役にする。左に一致するデータがなくても、右のデータはすべて残す。
#how='inner'：両方の表にあるデータだけを残す。どちらか片方にしかないデータは消える。
#how='outer'：両方の表のデータをすべて残す。一致しない部分は「欠損値(NaN)」で埋める。

#drop：不要なものを削除する。
#axis：処理する向き　axis=0：縦（列）　axis=1：横（行）

### ノック１９：クレンジングしたデータをダンプしよう

In [ ]:
dump_data = join_data[['purchase_date', 'purchase_month', 'item_name', 'item_price', '顧客名','かな', '地域', 'メールアドレス', '登録日']]
dump_data

#dump：「書き出す」「保存する」「1つにまとめて出す」

In [ ]:
dump_data.to_csv('dump_data.csv', index=False)

#df.to_csv('ファイル名',index=)：データをCSV形式で保存する。※分析結果をExcelで確認できるようにする。
#index=True：行番号もファイルに書き出す(保存する)。
#index=False：行番号を捨てて、中身だけ保存する。

### ノック２０：データを集計しよう

In [ ]:
import_data = pd.read_csv('dump_data.csv')
import_data

In [ ]:
byItem = import_data.pivot_table(index='purchase_month', columns='item_name', aggfunc='size', fill_value=0)
byItem

#pd.pivot_table：クロス集計表（Excelの表）を作る。※pd.pivot_table(表の名前,index=,columns=,values=,aggfunc=)
#「index=」行ラベル（縦軸）
#「columns=」列ラベル（横軸）
#「values=」値（計算対象）
#「aggfunc=」値の集計設定（基礎統計量）
#よく使う基礎統計量：sum（合計）,mean（平均値）,std（標準偏差）,median（中央値）,count（個数計算※欠損値(NaN)含まない）,size（行数計算※欠損値(NaN)含む）

#fill_value：空欄を埋めるための値の設定。※引数（設定項目）
#fill_value=0：空欄のセルは「NaN」ではなく「0」を入れる。

In [ ]:
byPrice = import_data.pivot_table(index='purchase_month', columns='item_name', values='item_price', aggfunc='sum', fill_value=0)
byPrice

In [ ]:
byCustomer = import_data.pivot_table(index='purchase_month', columns='顧客名', aggfunc='size', fill_value=0)
byCustomer

In [ ]:
byRegion = import_data.pivot_table(index='purchase_month', columns='地域', aggfunc='size', fill_value=0)
byRegion

In [ ]:
away_data = pd.merge(uriage_data, kokyaku_data, left_on='customer_name', right_on='顧客名', how='right')

#pd.merge()：結合。正規化されてバラバラになったデータを、分析しやすいように「逆正規化（合体）」させる。
#pd.merge(左の表,右の表, left_on=, right_on=, how=)

#left_on：左側の表(テーブル)の使用する列名を指定。
#right_on：右側の表(テーブル)の使用する列名を指定。
#※left_on, right_on はそれぞれの表(テーブル)で使う列名が異なるときに使う。

#how：表(テーブル)の結合方法を指定。（※よく使う結合方法↓）
#how='left'：左の表を主役にする。右に一致するデータがなくても、左のデータはすべて残す。
#how='right'：右の表を主役にする。左に一致するデータがなくても、右のデータはすべて残す。
#how='inner'：両方の表にあるデータだけを残す。どちらか片方にしかないデータは消える。
#how='outer'：両方の表のデータをすべて残す。一致しない部分は「欠損値(NaN)」で埋める。


In [ ]:
away_data[away_data['purchase_date'].isnull()][['顧客名','メールアドレス','登録日']]

#.isnull：「欠損値(NaN)セル」があるかどうかチェックする。→True：ある False：ない